# Actor-Critic

---
## 目的
Actor-Criticの仕組みを理解し，ゲームタスクを用いて強化学習を行う．
学習後のエージェントの可視化を行い，学習がうまくできているか確認を行う．

## Actor-Critic
Actor-Criticは強化学習における1手法で，推定した状態価値をもとに報酬をより多く得るように方策を更新するActorと，方策に対する状態価値を推定するCriticという2つの学習器を同時学習する手法です．
Actorは行動の確率分布から自身の方策により行動を選択します．Criticでは，計算したTD誤差に応じて状態価値関数$V(s)$を最適状態価値関数になるように更新します．

`policy_gradient.ipynb`のREINFORCEでは，実際に得られた収益$G_t$をそのまま勾配の重みとして用いていたため，分散が大きく学習が不安定になりやすいという問題がありました．Actor-Criticでは，Criticが出力する状態価値$V(s)$を用いて，収益の代わりにAdvantage（アドバンテージ）

$$
A(s_t,a_t)=G_t-V(s_t)
$$

を勾配の重みとして用いることで，分散を削減し学習を安定化させます．DCNNを用いたActor-Criticの手法では，ActorとCriticをそれぞれニューラルネットワーク（の一部）で表現し，同時に学習を行います．派生手法には，DDPG，A3C，UNREALなどがあります．


## 準備
下記のプログラムを実行して，実験に必要な追加ライブラリをインストールする．

In [ ]:
!pip install -q "gymnasium[atari,other]" ale-py

## モジュールのインポートとGPUの確認
はじめに必要なモジュールをインポートする．

今回はPyTorchに加えて，Pongを実行するためのシミュレータであるGymnasium（gymnasium）をインポートする．
そして，GPUが使用可能かどうかを確認する．

In [ ]:
import os
import zipfile
import time
import datetime
import random
import collections
import cv2
from base64 import b64encode

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML

import gymnasium as gym
import ale_py
from gymnasium.wrappers import RecordVideo

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical
import gdown

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Use device:', device)

## シード値の固定

In [ ]:
seed = 123
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms = True

## OpenAI GymによるPong環境の定義
[Gymnasium](https://gymnasium.farama.org/)は，様々な種類の環境を提供しているモジュールです．
今回は，Gymnasiumで利用可能なAtari2600のゲームであるPongを使用します．
Pongの行動数は6ですが，実質的な行動はパドルを上下どちらかに移動させる2種類のみです（詳細は`deep_q_network.ipynb`を参照）．

In [ ]:
gym.register_envs(ale_py)
env = gym.make('ALE/Pong-v5', frameskip=1, repeat_action_probability=0.0, full_action_space=False)

obs, info = env.reset(seed=seed, options=None)
print('observation space:', env.observation_space)
print('action space:', env.action_space)
print('initial observation:', obs.shape)

## 環境の前処理
Atari環境の学習を安定・効率化するため，`deep_q_network.ipynb`と同様の前処理を適用します．

* MaxAndSkipEnv：1ステップ実行毎に，4フレームゲームを進める（skip frame）
* FireResetEnv：エピソード（ゲーム）開始にFireを実行しなければ開始されない環境でのreset関数の設定
* ProcessFrame84：210×160のRGB画像を84×84のグレースケール画像に変換
* ImageToPyTorch：観測情報（画像）のshapeをHWC（高さ，幅，チャネル）からCHW（チャネル，高さ，幅）に変換
* ScaledFloatFrame：画像（0から255）を0.0から1.0の範囲で正規化
* BufferWrapper：観測情報を4フレームまとめて返す

In [ ]:
class MaxAndSkipEnv(gym.Wrapper):
    def __init__(self, env=None, skip=4):
        super(MaxAndSkipEnv, self).__init__(env)
        self._obs_buffer = collections.deque(maxlen=2)
        self._skip = skip

    def step(self, action):
        total_reward = 0.0
        done = None
        for _ in range(self._skip):
            obs, reward, done, truncated, info = self.env.step(action)
            self._obs_buffer.append(obs)
            total_reward += reward
            if done:
                break
        max_frame = np.max(np.stack(self._obs_buffer), axis=0)
        return max_frame, total_reward, done, truncated, info

class FireResetEnv(gym.Wrapper):
    def __init__(self, env=None):
        super(FireResetEnv, self).__init__(env)
        assert env.unwrapped.get_action_meanings()[1] == 'FIRE'
        assert len(env.unwrapped.get_action_meanings()) >= 3

    def step(self, action):
        return self.env.step(action)

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        obs, _, done, truncated, _ = self.env.step(1)
        if done or truncated:
            obs, info = self.env.reset(**kwargs)
        obs, _, done, truncated, _ = self.env.step(2)
        if done or truncated:
            obs, info = self.env.reset(**kwargs)
        return obs, info

class ProcessFrame84(gym.ObservationWrapper):
    def __init__(self, env=None):
        super(ProcessFrame84, self).__init__(env)
        self.observation_space = gym.spaces.Box(low=0, high=255, shape=(84, 84, 1), dtype=np.uint8)

    def observation(self, obs):
        return ProcessFrame84.process(obs)

    @staticmethod
    def process(frame):
        if frame.size == 210 * 160 * 3:
            img = np.reshape(frame, [210, 160, 3]).astype(np.float32)
        elif frame.size == 250 * 160 * 3:
            img = np.reshape(frame, [250, 160, 3]).astype(np.float32)
        else:
            assert False, "Unknown resolution."
        img = img[:, :, 0] * 0.299 + img[:, :, 1] * 0.587 + img[:, :, 2] * 0.114
        resized_screen = cv2.resize(img, (84, 110), interpolation=cv2.INTER_AREA)
        x_t = resized_screen[18:102, :]
        x_t = np.reshape(x_t, [84, 84, 1])
        return x_t.astype(np.uint8)

class ImageToPyTorch(gym.ObservationWrapper):
    def __init__(self, env):
        super(ImageToPyTorch, self).__init__(env)
        old_shape = self.observation_space.shape
        self.observation_space = gym.spaces.Box(low=0.0, high=1.0, shape=(old_shape[-1], old_shape[0], old_shape[1]),
                                                dtype=np.float32)

    def observation(self, observation):
        return np.moveaxis(observation, 2, 0)

class ScaledFloatFrame(gym.ObservationWrapper):
    def observation(self, obs):
        return np.array(obs).astype(np.float32) / 255.0

class BufferWrapper(gym.ObservationWrapper):
    def __init__(self, env, n_steps, dtype=np.float32):
        super(BufferWrapper, self).__init__(env)
        self.dtype = dtype
        old_space = env.observation_space
        self.observation_space = gym.spaces.Box(old_space.low.repeat(n_steps, axis=0),
                                                old_space.high.repeat(n_steps, axis=0), dtype=dtype)

    def reset(self, **kwargs):
        self.buffer = np.zeros_like(self.observation_space.low, dtype=self.dtype)
        obs, info = self.env.reset(**kwargs)
        return self.observation(obs), info

    def observation(self, observation):
        self.buffer[:-1] = self.buffer[1:]
        self.buffer[-1] = observation
        return self.buffer

### 前処理の適用
環境に対して必要となる前処理を適用します．

In [ ]:
gym.register_envs(ale_py)
env = gym.make('ALE/Pong-v5', frameskip=1, repeat_action_probability=0.0, full_action_space=False)

env = MaxAndSkipEnv(env)
env = FireResetEnv(env)
env = ProcessFrame84(env)
env = ImageToPyTorch(env)
env = BufferWrapper(env, 4)
env = ScaledFloatFrame(env)

## ネットワーク構造
Actor-Criticのネットワークを定義します．畳み込み層3層からなる共通の特徴抽出部と，そこから分岐する全結合層2層のブランチを2つ持つネットワークとします．それぞれのブランチをPolicy branchとValue branchとしていて，Policy branchのネットワークからはエージェントの行動に対するスコア（logit）を，Value branchのネットワークからは状態価値を出力します．

In [ ]:
class ActorCritic(nn.Module):
    def __init__(self, input_shape, n_actions):
        super(ActorCritic, self).__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(input_shape[0], 32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU()
        )

        conv_out_size = self._get_conv_out(input_shape)
        self.policy = nn.Sequential(
            nn.Linear(conv_out_size, 512),
            nn.ReLU(),
            nn.Linear(512, n_actions)
        )
        self.value = nn.Sequential(
            nn.Linear(conv_out_size, 512),
            nn.ReLU(),
            nn.Linear(512, 1)
        )

    def _get_conv_out(self, shape):
        o = self.conv(torch.zeros(1, *shape))
        return int(np.prod(o.size()))

    def forward(self, x):
        conv_out = self.conv(x).view(x.size(0), -1)
        return self.value(conv_out), self.policy(conv_out)

## エージェントの定義
エージェントが環境に対して確率的に行動し，`num_steps`ステップ分の状態価値・対数確率・報酬・エントロピーをそれぞれ収集します．

`action_train`関数では，ネットワークへ環境情報（画像）を入力して行動の確率と状態価値を受け取り，行動の確率をもとに行動をサンプリングします．また，方策の探索を促すため，行動の確率分布のエントロピーも同時に記録しておきます．

In [ ]:
class Agent:
    def __init__(self, model, env, device, seed=None):
        self.model = model
        self.env = env
        self.device = device
        if seed is not None:
            self.env.reset(seed=seed)  # 最初に一度だけ乱数系列をシード固定する（以降のエピソードでは多様な初期状態を用いる）
        self.state, _ = self.env.reset()
        self.done = False
        self.clear_actions()

    def action_train(self):
        state_v = torch.as_tensor(self.state).unsqueeze(0).to(self.device)
        value, logit = self.model(state_v)

        prob = F.softmax(logit, dim=1)
        log_prob = F.log_softmax(logit, dim=1)
        entropy = -(log_prob * prob).sum(1)

        action = prob.multinomial(1).detach()
        log_prob_a = log_prob.gather(1, action)

        next_state, reward, terminated, truncated, info = self.env.step(action.item())
        self.done = terminated or truncated
        reward = max(min(reward, 1), -1)  # Reward clipping

        self.values.append(value)
        self.log_probs.append(log_prob_a)
        self.rewards.append(reward)
        self.entropies.append(entropy)

        self.state = next_state
        if self.done:
            self.state, _ = self.env.reset()

        return reward

    def clear_actions(self):
        self.values = []
        self.log_probs = []
        self.rewards = []
        self.entropies = []

## Lossの計算
Actor-CriticのLoss計算はそれぞれ独立して行います．Valueでは最適状態価値を出力するように学習を行います．Loss計算は，次状態の推定の価値と実際に起こした行動から得られる価値の差を0にするように状態価値関数$V$を更新していくTD誤差を用いて更新します．ここで，$r$は報酬値で$\gamma$は割引率です．

$$
L_{v}=(r+{\gamma}V(s_{t+1})-V(s_t))^2
$$

Policyでは，最適な行動の確率を上げるように学習を行います．勾配の計算には，方策勾配定理を利用します．方策勾配法では，targetとなる部分を収益$G_t$そのものを用いていましたが，Actor-Criticの場合は，Valueの結果を利用したAdvantage $A(s)$とします．$A(s)$は次状態の推定の価値と実際に起こした行動から得られる価値の差であり，Valueでの更新で使用したLossと同じです．方策$\pi$のLossにそのまま用います．ここで，$H$はエントロピーであり，$\beta$はエントロピーの正則化項です．

また，$A(s)$の計算でValueの出力を使用しているため，勾配が流れてしまわないように注意してください．

$$
A(s)=r+{\gamma}V(s_{t+1})-V(s_t)
$$
$$
L_p=-\log(\pi(a|s))A(s)-\beta H(\pi)
$$

Lossの計算を行う関数を定義します．`calc_loss`関数ではPolicyとValueの両方のLossを計算します．`R`は，`num_steps`分のrolloutの末尾（あるいはエピソード終了時）における状態価値（ブートストラップ値）で，そこから逆順に割引報酬和を累積させることで各時刻のAdvantageを計算します．

In [ ]:
def calc_loss(agent, R, gamma, entropy_coef, device):
    policy_loss = 0
    value_loss = 0

    for i in reversed(range(len(agent.rewards))):
        R = gamma * R + agent.rewards[i]
        advantage = R - agent.values[i]
        value_loss = value_loss + 0.5 * advantage.pow(2)

        policy_ad = advantage.detach()
        policy_loss = policy_loss - agent.log_probs[i] * policy_ad - entropy_coef * agent.entropies[i]

    return policy_loss, value_loss

## 学習
Actor-Criticを用いて学習を行います．学習環境はAtari環境のPongゲーム環境を用います．
`num_steps`ステップ分のrolloutを集めるたびにネットワークを更新する，n-step Actor-Criticとして実装します．

Pongは報酬が疎な環境であり，人間並みのスコアに到達するには非常に多くのフレーム数（数百万フレーム以上）が必要です．ここでは，学習の仕組みと損失・報酬の推移を確認できる範囲のフレーム数で実行します．

In [ ]:
GAMMA = 0.99
LEARNING_RATE = 1e-4
ENTROPY_COEF = 0.01
VALUE_COEF = 0.5
num_steps = 20        # rolloutのステップ数
num_frame = 200000    # 収束にはこの数十倍以上のフレーム数が必要
MODEL_PATH = 'actor_critic.pt'

acnet = ActorCritic(env.observation_space.shape, env.action_space.n).to(device)
optimizer = optim.Adam(acnet.parameters(), lr=LEARNING_RATE)
agent = Agent(acnet, env, device, seed)

frame_idx = 0
episode_reward = 0.0
total_rewards = []
record_reward = []
record_step = []
best_mean_reward = None

acnet.train()
ts = time.time()
while frame_idx < num_frame:
    agent.clear_actions()
    for step in range(num_steps):
        reward = agent.action_train()
        episode_reward += reward
        frame_idx += 1
        if agent.done:
            total_rewards.append(episode_reward)
            episode_reward = 0.0
            mean_reward = np.mean(total_rewards[-20:])
            record_reward.append(mean_reward)
            record_step.append(frame_idx)
            if len(total_rewards) % 5 == 0:
                print('Frame {0}/{1}: episode {2}, mean reward {3:.3f}, time {4}'.format(
                    frame_idx, num_frame, len(total_rewards), mean_reward, datetime.timedelta(seconds=time.time() - ts)))

            # 平均報酬が更新されたら最良モデルとして保存する
            if best_mean_reward is None or best_mean_reward <= mean_reward:
                torch.save(acnet.state_dict(), MODEL_PATH)
                if best_mean_reward is not None:
                    print('Best mean reward updated {0:.3f} -> {1:.3f}, model saved'.format(best_mean_reward, mean_reward))
                best_mean_reward = mean_reward
            break

    R = torch.zeros(1, 1, device=device)
    if not agent.done:
        with torch.no_grad():
            value, _ = acnet(torch.as_tensor(agent.state).unsqueeze(0).to(device))
        R = value

    policy_loss, value_loss = calc_loss(agent, R, GAMMA, ENTROPY_COEF, device)

    optimizer.zero_grad()
    (policy_loss + VALUE_COEF * value_loss).backward()
    optimizer.step()

## 学習時の平均スコアの推移
横軸フレーム数，縦軸平均スコアとしたグラフを描画してみます．

In [ ]:
fig = plt.figure()
plt.plot(record_step, record_reward, color="red")
plt.grid()
plt.xlabel("step")
plt.ylabel("mean reward")
plt.savefig("./actor_critic_step_per_reward.png")
plt.show()

## 評価

学習したネットワーク（エージェント）を確認してみます．学習時に保存した学習済みモデル（`MODEL_PATH`）を読み込み，テストに使用します．`RecordVideo`ラッパーを用いて，エージェントによるゲームプレイを動画として記録し，Notebook上で再生します．評価時は，行動をサンプリングせず，最も確率の高い行動を選択します．

### 学習済みモデルのダウンロード

プログラム演習中に十分な学習時間が確保できない場合は，下記のプログラムを実行して学習済みモデルをダウンロードし使用してください．

In [ ]:
if not os.path.isdir('./actor_critic_model'):
    gdown.download(id='1pbMulKYjfu38g7ZpH39qzqTlf9WGV4VQ', output='actor_critic_model.zip', quiet=False)
    with zipfile.ZipFile('actor_critic_model.zip') as f:
        f.extractall('./')

In [ ]:
gym.register_envs(ale_py)
eval_env = gym.make('ALE/Pong-v5', frameskip=1, repeat_action_probability=0.0, full_action_space=False, render_mode='rgb_array')
eval_env = MaxAndSkipEnv(eval_env)
eval_env = FireResetEnv(eval_env)
eval_env = ProcessFrame84(eval_env)
eval_env = ImageToPyTorch(eval_env)
eval_env = BufferWrapper(eval_env, 4)
eval_env = ScaledFloatFrame(eval_env)
eval_env = RecordVideo(eval_env, './video/actor_critic', episode_trigger=lambda x: True)

# 学習済みモデルの読み込み
MODEL_PATH = 'actor_critic_model/actor_critic.pt'
assert os.path.isfile(MODEL_PATH), '{0} が見つかりません．学習セルを実行してモデルを保存してから実行してください．'.format(MODEL_PATH)
acnet = ActorCritic(eval_env.observation_space.shape, eval_env.action_space.n).to(device)
acnet.load_state_dict(torch.load(MODEL_PATH, map_location=device))
acnet.eval()

state, info = eval_env.reset()
done = False
with torch.no_grad():
    while not done:
        state_v = torch.as_tensor(state).unsqueeze(0).to(device)
        _, logit = acnet(state_v)
        action = torch.argmax(logit, dim=1).item()

        state, reward, terminated, truncated, info = eval_env.step(action)
        done = terminated or truncated

eval_env.close()

mp4 = open('./video/actor_critic/rl-video-episode-0.mp4', 'rb').read()
data_url = 'data:video/mp4;base64,' + b64encode(mp4).decode()
HTML(f"""
<video width="320" height="420" controls>
      <source src="{data_url}" type="video/mp4">
</video>""")

## 課題

1. `num_steps`（rolloutのステップ数）を変えて，学習の安定性がどのように変わるか確認してみましょう．
2. `ENTROPY_COEF`（エントロピー正則化の重み）を変えて，探索の様子がどのように変わるか確認してみましょう．
3. Pong以外のゲームで学習してみましょう．
    * `gym.make()`での環境の指定を変更することで，任意の環境で学習できます．
    * 指定できる環境は，`ALE/Breakout-v5`や`ALE/MsPacman-v5`などがあります．詳しくは[ドキュメント](https://ale.farama.org/environments/)をチェックしてください．